In [20]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [21]:
from langchain_openai import ChatOpenAI
from langchain_classic.memory  import ConversationBufferMemory,ConversationBufferWindowMemory,ConversationSummaryBufferMemory
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

model = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    model="gpt-5.1",
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful chatbot"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{message}"),
    ]
)

buffer_memory = ConversationBufferMemory(return_messages=True)

wd_memory = ConversationBufferWindowMemory(
    return_messages=True,
    k=3,
)

sum_memory = ConversationSummaryBufferMemory(
    llm=model,
    max_token_limit=100,
    return_messages=True,
)

def load_buffer_memory(_):
    return buffer_memory.load_memory_variables({})["history"]

def load_wd_memory(_):
    return wd_memory.load_memory_variables({})["history"]   

def load_sum_memory(_):
    return sum_memory.load_memory_variables({})["history"]

In [ ]:
# 10-turn conversation test for memory
test_conversations = [
    {"input": "Hi, my name is Cuong.", "output": "Nice to meet you, Cuong!"},
    {"input": "I live in Hanoi.", "output": "Great, you live in Hanoi."},
    {"input": "My favorite language is Python.", "output": "Awesome, Python is your favorite language."},
    {"input": "I work as a backend developer.", "output": "Got it, you are a backend developer."},
    {"input": "I enjoy building APIs with FastAPI.", "output": "Cool, you like building APIs with FastAPI."},
    {"input": "My favorite hobby is running.", "output": "Nice, running is your favorite hobby."},
    {"input": "I ran 5km yesterday.", "output": "Great job! You ran 5km yesterday."},
    {"input": "I also like reading AI newsletters.", "output": "Noted, you also enjoy reading AI newsletters."},
    {"input": "Can you summarize what you know about me so far?", "output": "Sure! You have shared quite a bit with me over our conversation."},    {"input": "What city do I live in and what is my favorite language?", "output": "You live in Hanoi and your favorite language is Python."},
]

# Clear all memories before test
buffer_memory.clear()
wd_memory.clear()
sum_memory.clear()

def add_message(input, output):
    buffer_memory.save_context({"input": input}, {"output": output})
    wd_memory.save_context({"input": input}, {"output": output})
    sum_memory.save_context({"input": input}, {"output": output})

for turn, message in enumerate(test_conversations, start=1):
    user_text = message["input"]
    ai_text = message["output"]
    add_message(user_text, ai_text)


# Content of memories after 10 turns
bff_history = buffer_memory.load_memory_variables({})["history"]
print(f"Total messages in buffer_memory: {len(bff_history)}")
print(f"Buffer_memory: {bff_history}")
wd_history = wd_memory.load_memory_variables({})["history"]
print(f"Total messages in wd_memory: {len(wd_history)}")   
print(f"WD_memory: {wd_history}")

sum_history = sum_memory.load_memory_variables({})["history"]
print(f"Total messages in sum_memory: {len(sum_history)}")
print(f"Sum_memory: {sum_history}")


Total messages in buffer_memory: 20
Buffer_memory: [HumanMessage(content='Hi, my name is Cuong.', additional_kwargs={}, response_metadata={}), AIMessage(content='Nice to meet you, Cuong!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='I live in Hanoi.', additional_kwargs={}, response_metadata={}), AIMessage(content='Great, you live in Hanoi.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='My favorite language is Python.', additional_kwargs={}, response_metadata={}), AIMessage(content='Awesome, Python is your favorite language.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='I work as a backend developer.', additional_kwargs={}, response_metadata={}), AIMessage(content='Got it, you are a backend developer.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='I

In [ ]:
buffer_chain = RunnablePassthrough.assign(history=load_buffer_memory) | prompt | model
wd_chain = RunnablePassthrough.assign(history=load_wd_memory) | prompt | model
sum_chain = RunnablePassthrough.assign(history=load_sum_memory) | prompt | model

# Test memory still save first messages after 10 turns
question = "What is my name?"
buffer_response = buffer_chain.invoke({"message": question})
print(f"Buffer_memory response: {buffer_response}")
wd_response = wd_chain.invoke({"message": question})    
print(f"WD_memory response: {wd_response}")
sum_response = sum_chain.invoke({"message": question})
print(f"Sum_memory response: {sum_response}")


Buffer_memory response: content='Your name is Cuong.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 293, 'total_tokens': 309, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.1-2025-11-13', 'system_fingerprint': None, 'id': 'chatcmpl-DNIPUUVYcosJ4X0bMgBZrfPcJsPzq', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019d2525-257c-7d31-808a-6e817dced441-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 293, 'output_tokens': 16, 'total_tokens': 309, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
WD_memory response: content='I don’t know your name—you haven’t told me that yet.' additional_kwargs={'refusal': None} response_metadata={'toke

In [ ]:
# count tokens in memory
bff_history = buffer_memory.load_memory_variables({})["history"]
wd_history = wd_memory.load_memory_variables({})["history"]
sum_history = sum_memory.load_memory_variables({})["history"]

print(f"buffer_memory tokens: {model.get_num_tokens_from_messages(bff_history)}")
print(f"wd_memory tokens:     {model.get_num_tokens_from_messages(wd_history)}")
print(f"sum_memory tokens:    {model.get_num_tokens_from_messages(sum_history)}")

buffer_memory tokens: 256
wd_memory tokens:     93
sum_memory tokens:    243


In [ ]:
memory = ConversationSummaryBufferMemory(
    llm=model,
    max_token_limit=120,
    return_messages=True,
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a Python programming tutor professor."),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}"),
    ]
)

def load_memory(_):
    return memory.load_memory_variables({})["history"]

chain = RunnablePassthrough.assign(history=load_memory) | prompt | model

def invoke_chain(question):
    result = chain.invoke({"question": question})
    memory.save_context(
        {"input": question},
        {"output": result.content},
    )
    print(result)

invoke_chain("What is a Python decorator?")
invoke_chain("Can you give me an example of a Python decorator?")   
invoke_chain("What is the difference between a function and a method in Python?")

load_memory(None)  


content='A Python decorator is a callable (usually a function) that takes another function (or method or class) as input, adds or changes some behavior, and returns a new function (or object), without you having to modify the original function’s code.\n\nConceptually:\n\n```python\ndef decorator(fn):\n    def wrapper(*args, **kwargs):\n        # code before\n        result = fn(*args, **kwargs)\n        # code after\n        return result\n    return wrapper\n```\n\nUsing a decorator with the `@` syntax:\n\n```python\ndef debug(fn):\n    def wrapper(*args, **kwargs):\n        print(f"Calling {fn.__name__} with {args} {kwargs}")\n        result = fn(*args, **kwargs)\n        print(f"{fn.__name__} returned {result}")\n        return result\n    return wrapper\n\n@debug                 # same as: add = debug(add)\ndef add(a, b):\n    return a + b\n\nadd(2, 3)\n```\n\nHere, `@debug` “wraps” `add`. When you call `add(2, 3)`, you’re actually calling `wrapper`, which logs information and then

[SystemMessage(content='The human asks what a Python decorator is. The AI explains that a decorator is a callable that takes another function (or method or class), adds or changes behavior, and returns a new callable without modifying the original code. It shows the typical wrapper pattern, demonstrates using the `@` syntax with a `debug` example that logs calls and results, and lists common uses such as logging/debugging, auth checks, caching, timing, and enforcing types or conditions, offering to walk through writing one step by step. The human then asks for an example, and the AI provides a concrete `debug` decorator implementation using `functools.wraps`, shows how to apply it with `@debug` to `add` and `greet` functions, demonstrates the resulting debug output, and explains in detail how `@debug` is equivalent to `add = debug(add)` and how the wrapper function intercepts, logs, calls the original function, and returns its result. The human then asks for the difference between a fu